In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
import joblib

In [ ]:
df=pd.read_csv("C:\\Users\\New folder\\bluebook-for-bulldozers\\TrainAndValid.csv",
               low_memory=False)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 412698 entries, 0 to 412697
Data columns (total 53 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   SalesID                   412698 non-null  int64  
 1   SalePrice                 412698 non-null  float64
 2   MachineID                 412698 non-null  int64  
 3   ModelID                   412698 non-null  int64  
 4   datasource                412698 non-null  int64  
 5   auctioneerID              392562 non-null  float64
 6   YearMade                  412698 non-null  int64  
 7   MachineHoursCurrentMeter  147504 non-null  float64
 8   UsageBand                 73670 non-null   object 
 9   saledate                  412698 non-null  object 
 10  fiModelDesc               412698 non-null  object 
 11  fiBaseModel               412698 non-null  object 
 12  fiSecondaryDesc           271971 non-null  object 
 13  fiModelSeries             58667 non-null   o

In [ ]:
df["saledate"] = pd.to_datetime(df["saledate"])

In [ ]:
df["saleYear"] = df["saledate"].dt.year
df["saleMonth"] = df["saledate"].dt.month
df["saleDay"] = df["saledate"].dt.day
df["saleDayOfWeek"] = df["saledate"].dt.dayofweek


In [ ]:
df.drop("saledate", axis=1, inplace=True)

In [ ]:
X=df.drop("SalePrice", axis=1)
y=df["SalePrice"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
cat=X.select_dtypes(include=["object"]).columns
num=X.select_dtypes(exclude=["object"]).columns

In [ ]:
num_pipeline=Pipeline([("imputer", SimpleImputer(strategy="median"))])

In [ ]:
cat_pipeline=Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                       ("onehot", OneHotEncoder(handle_unknown="ignore"))])

In [ ]:
preprocessor=ColumnTransformer([("num", num_pipeline, num),
                              ("cat", cat_pipeline, cat)])

In [ ]:
model=Pipeline([("preprocessor", preprocessor),
                ("model", RandomForestRegressor(n_jobs=-1, random_state=42))])

In [ ]:
paras_dict = {
    "model__n_estimators": [100, 200, 500],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10],
}

In [36]:
rs_model = RandomizedSearchCV(model, param_distributions=paras_dict, n_iter=3, cv=2, verbose=2, random_state=42)

In [37]:
rs_model.fit(X_train, y_train)

Fitting 2 folds for each of 3 candidates, totalling 6 fits


KeyboardInterrupt: 

In [38]:
y_pred=rs_model.predict(X_test)
score=rs_model.score(X_test, y_test)
print(score)

NotFittedError: This RandomizedSearchCV instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [ ]:
mae=mean_absolute_error(y_test, y_pred)
print(mae)  
joblib.dump(rs_model, "bulldozer_price_model.pkl")